# Fase 4. Modelos logísticos multinivel

**Tesis** · Desigualdades socioeconómicas, étnicas y territoriales en el bajo peso al nacer
en Colombia: análisis multinivel y predictivo de las estadísticas vitales, 1998-2024
**Autor** · Gregory Jesús Meléndez · **Director** · Lihki José Rubio Ortega
**Corresponde a** · Capítulo 4, Sección 4.4.4 del manuscrito

| | |
|---|---|
| **Entra** | `intermedios/muestra_armonizada.parquet` |
| **Sale** | `intermedios/modelos_multinivel.pkl`, tres tablas, dos figuras |
| **Tiempo** | **entre 20 y 50 minutos.** Es la parte más pesada de la tesis |

---

## Qué responde esta fase

Dos preguntas que las fases anteriores no pueden contestar:

1. **Cuánta de la variación en el bajo peso al nacer es territorial**, y no individual. El
   índice de concentración de la Fase 3 mide desigualdad entre personas ordenadas por
   posición social; no dice nada sobre cuánto depende el riesgo del lugar donde se vive.
2. **Cuál es el efecto de cada determinante social una vez descontado el territorio.** Las
   prevalencias crudas de la Fase 1 y el índice de la Fase 3 confunden el efecto de la
   posición social con el del lugar, porque las madres de distinta educación no se
   distribuyen por igual entre municipios.

## Qué NO responde

No establece causalidad. Un modelo multinivel controla la confusión por las variables que se
le dan, y solo por esas. Quedan fuera la nutrición materna, el tabaquismo, la exposición
laboral y todo lo que las estadísticas vitales no recogen.

Y no resuelve el problema que atraviesa todo el trabajo: **el desenlace es bajo peso
registrado**, no bajo peso ocurrido. El modelo estima asociaciones con lo que quedó
registrado.

## La estructura de los datos

Un nacimiento ocurre en un municipio, y cada municipio pertenece a un departamento. Eso es
un anidamiento estricto de tres niveles, y tratarlo como si las observaciones fueran
independientes tendría dos consecuencias, las dos malas:

- **Los errores estándar saldrían demasiado estrechos.** Dos nacimientos del mismo municipio
  se parecen más entre sí que dos de municipios distintos, de modo que 17 millones de
  nacimientos no aportan 17 millones de piezas de información independiente.
- **Se perdería el objeto mismo del análisis.** Cuánta variación es territorial es una de las
  preguntas de la tesis, no un estorbo que haya que corregir.

## Los cinco modelos NO son cinco técnicas distintas

**Son la misma regresión, ajustada cinco veces.** Conviene dejarlo claro antes de nada,
porque es la confusión más fácil de tener al leer el capítulo.

La técnica es **una sola**: regresión logística multinivel de interceptos aleatorios. Lo único
que cambia entre M0 y M4 es **qué variables entran**. Mismo tipo de regresión, mismos datos,
misma estructura jerárquica, misma estimación.

Es como pesar algo en la misma balanza cinco veces, retirando una capa cada vez. La balanza no
cambia; **la diferencia entre pesadas** es lo que dice cuánto pesaba cada capa. Por eso el
resultado de esta fase no está solo en cada modelo, sino en **la diferencia entre uno y el
siguiente**.

Es lo que el Capítulo 3 llama **estrategia de modelos anidados**: cada modelo contiene a todos
los anteriores más un bloque nuevo.

**No confundirlos con los siete modelos de clasificación de la Fase 7.** Aquéllos sí son siete
técnicas distintas, comparadas por su capacidad predictiva. Éstos son cinco ajustes de una
misma especificación, comparados entre sí para separar confusión de mediación. El propósito es
opuesto: explicar frente a predecir.

## Los cinco modelos, y por qué en este orden

| | Qué añade | Qué estima |
|---|---|---|
| **M0** | nada, solo los interceptos aleatorios | cuánta variación es territorial **antes** de cualquier ajuste |
| **M1** | determinantes sociales | el **efecto total** de la posición social |
| **M2** | edad materna, paridad y multiplicidad | el **efecto independiente** de cada determinante social |
| **M3** | reconocimiento étnico | el efecto social **descontada la composición étnica** |
| **M4** | consultas prenatales | el **efecto directo**, descontado lo que pasa por la atención prenatal |

El orden no es arbitrario: **cada paso responde a una pregunta distinta, no a una versión
mejor controlada de la misma pregunta.**

**M2 es el modelo que responde al tercer objetivo específico**, que pide estimar el efecto
independiente de cada determinante social «ajustados por edad materna, paridad y
multiplicidad del embarazo». Las tres son covariables de ajuste: influyen en el peso al nacer
por vías biológicas y se distribuyen de forma desigual entre grupos sociales, de modo que no
controlarlas dejaría confusión dentro del coeficiente social.

**M4 es distinto y no es «M3 mejor controlado».** Añade un **mediador**, no un confusor, y
por eso estima otra cantidad. La distinción entre los dos, que es lo que separa M4 de los anteriores, se
explica en su bloque y es la decisión metodológica que un jurado va a examinar.

## Advertencia computacional

El tiempo crece con el número de celdas binomiales y con el número de efectos fijos. Los
órdenes de magnitud medidos:

| Modelo | Celdas | Efectos fijos | Tiempo |
|---|---:|---:|---|
| M0 | 1.157 | 1 | segundos |
| M1 | 124.635 | 13 | ~75 s |
| M2 | 629.032 | 20 | varios minutos |
| M3, M4 | menos celdas, más términos | | varios minutos |

**Cada modelo se persiste en `.pkl` en cuanto termina**, de modo que un fallo en el último no
obliga a repetir los anteriores. Quien lo ejecute debe dejarlo correr entero y no
interrumpirlo a mitad; si lo interrumpe, los modelos ya ajustados siguen en el archivo.

## 0. Entorno y el estimador

**Qué hacemos.** Importamos `src/multinivel.py`, que contiene el estimador.

**Por qué el proyecto tiene un estimador propio y no usa una biblioteca.** En Python no hay
una implementación de modelo lineal generalizado mixto que soporte esta escala.
`statsmodels` ofrece `BinomialBayesMixedGLM`, que usa aproximación variacional y construye
una matriz de diseño con **una columna por municipio**: con 1.157 municipios y millones de
filas, no cabe en memoria. Las alternativas serias —`lme4` en R, GLLAMM— exigen salir del
proyecto y romper la reproducibilidad de un solo entorno.

El modelo que esta tesis necesita, en cambio, es el caso más simple de la familia: **solo
interceptos aleatorios**, sin pendientes aleatorias. Para ese caso la verosimilitud tiene
estructura suficiente para resolverse de forma directa.

**Las tres decisiones de cómputo, y por qué son exactas y no atajos.**

**Agregación binomial.** Todas las covariables son categóricas, de modo que los 17 millones
de nacimientos se reducen a las combinaciones distintas de municipio, año y covariables.
Dentro de una combinación todos los nacimientos comparten el mismo predictor lineal, y la
suma de sus verosimilitudes de Bernoulli **es** la verosimilitud binomial de la celda. La
función objetivo es la misma salvo una constante que no depende de los parámetros.

**Aproximación de Laplace.** La integral sobre los efectos aleatorios se aproxima por una
gaussiana centrada en su modo. Es lo que hace `lme4` por defecto. Su exactitud depende de
cuánta información aporta cada agrupamiento, y aquí cada municipio tiene una mediana de
3.871 nacimientos. Se contrastó contra cuadratura adaptativa de Gauss-Hermite con 15 nodos:
coinciden **hasta el quinto decimal**, también en el peor caso de celdas con un solo
nacimiento.

**Hessiano en forma cerrada.** Como cada municipio pertenece a exactamente un departamento,
la matriz de segundas derivadas tiene estructura de flecha y su complemento de Schur es
diagonal. El paso de Newton y el determinante se calculan en tiempo lineal, sin construir ni
invertir ninguna matriz grande.

**Cómo se validó el estimador.** Sobre datos simulados con varianzas conocidas, en cinco
réplicas de 1.157 municipios en 33 departamentos: el sesgo del coeficiente de la covariable
fue **+0,0004**, el de la desviación municipal **+0,003** y el de la departamental
**+0,001**. Cuando la varianza departamental es cero en la simulación, el estimador la sitúa
en 0,009. Los efectos aleatorios se recuperan con correlación de 0,97 frente a los
verdaderos.

In [2]:
# ---------------------------------------------------------------------------
# Entorno
# ---------------------------------------------------------------------------

import sys, pickle
from pathlib import Path

def _encontrar_raiz(inicio: Path) -> Path:
    for candidata in [inicio, *inicio.parents]:
        if (candidata / "src" / "config.py").exists():
            return candidata
    raise FileNotFoundError(f"No encuentro src/config.py subiendo desde {inicio}")

RAIZ = _encontrar_raiz(Path.cwd())
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import SEMILLA, CONTROL, DIR_INTERMEDIOS
from datos import leer_intermedio
from util import cronometro, guardar_tabla, versiones
from graficos import estilo, guardar_figura
import armonizar
import multinivel

estilo()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

VERSION = "304768b2"
print(f"version en memoria: {VERSION}")
try:
    import hashlib, json as _json
    _c = _json.load(open(RAIZ / "notebooks" / "04_multinivel.ipynb",
                         encoding="utf-8"))["cells"]
    _t = "\n".join("".join(c["source"]) for c in _c).replace(VERSION, "")
    _d = hashlib.sha1(_t.encode("utf-8")).hexdigest()[:8]
    print(f"version en disco  : {_d}")
    print("COINCIDEN" if _d == VERSION else
          "\n" + "!"*60 + "\nNO COINCIDEN: la pestana esta desactualizada.\n"
          "Ctrl+Shift+P -> 'File: Revert File'. No interpretes esta corrida.\n" + "!"*60)
except FileNotFoundError:
    print("(archivo en disco no encontrado)")

# --- Carga -----------------------------------------------------------------
# Las tres ultimas —EDAD_MADRE, N_HIJOSV y MUL_PARTO— son las que M2 usa para
# construir EDAD3, PARIDAD4 y MULTIPLE3. Si faltan aqui, el notebook lee bien y
# no falla hasta la celda de M2, con un KeyError que parece un problema de la
# base y es solo una columna sin pedir.
COLUMNAS = ["ANIO", "BPN", "EDUC4_MADRE", "ESTCIV5", "SEG4", "AREA_RES",
            "IDPERTET", "NUMCONSUL", "DPTO_RES", "MUNI_RES",
            "EDAD_MADRE", "N_HIJOSV", "MUL_PARTO"]

with cronometro("lectura"):
    muestra = leer_intermedio("muestra_armonizada", columnas=COLUMNAS)

print(f"\nmuestra: {len(muestra):,} registros")
print(f"municipios: {muestra['MUNI_RES'].nunique():,}   "
      f"departamentos: {muestra['DPTO_RES'].nunique()}")

# --- Etiquetas, para que las tablas de resultados sean legibles -------------
ETIQUETAS = {
    "EDUC4_MADRE": armonizar.etiquetas_de("EDUC4"),
    "ESTCIV5": armonizar.etiquetas_de("ESTCIV5"),
    "SEG4": armonizar.etiquetas_de("SEG4"),
    "AREA_RES": {"1": "Cabecera", "2": "Centro poblado", "3": "Rural disperso"},
    "IDPERTET": {"1": "Indigena", "2": "Rom", "3": "Raizal",
                 "4": "Palenquero", "5": "Afrodescendiente",
                 "6": "Ninguna de las anteriores"},
}

# Donde se van guardando los modelos segun terminan.
MODELOS = {}
RUTA_MODELOS = DIR_INTERMEDIOS / "modelos_multinivel.pkl"

# --- Comprobacion de que estan todas las columnas que el notebook usa -------
# POR QUE AQUI: un KeyError en la celda de M2, quince minutos despues de
# empezar, no dice que el problema es la lista de arriba. Aqui si.
_NECESARIAS = ["EDAD_MADRE", "N_HIJOSV", "MUL_PARTO", "NUMCONSUL", "IDPERTET"]
_faltan = [c for c in _NECESARIAS if c not in muestra.columns]
if _faltan:
    raise KeyError(
        f"Faltan columnas en COLUMNAS: {_faltan}. "
        "M2 construye EDAD3, PARIDAD4 y MULTIPLE3 a partir de las tres "
        "primeras; M4 usa NUMCONSUL y M3 usa IDPERTET.")
print("\ncolumnas necesarias para M0 a M4: todas presentes")

version en memoria: 304768b2
version en disco  : 304768b2
COINCIDEN
[inicio] lectura
[fin]    lectura  ->  4.7 s

muestra: 17,376,860 registros
municipios: 1,157   departamentos: 33

columnas necesarias para M0 a M4: todas presentes


**Cómo se lee esta salida.** Debe decir **17.376.860** registros, **1.157** municipios y
**33** departamentos. Si falta `muestra_armonizada`, hay que correr antes
`01b_armonizacion.ipynb`.

## 1. Modelo vacío: cuánta variación es territorial

**Qué hacemos.** Ajustamos el modelo sin ninguna covariable: solo el intercepto general y los
interceptos aleatorios de municipio y departamento.

$$\text{logit}(\pi_{ijk})=\beta_0+u_k+v_{jk}$$

con $u_k\sim N(0,\sigma^2_{dpto})$ y $v_{jk}\sim N(0,\sigma^2_{mun})$.

**Por qué se empieza por aquí.** Este modelo es la referencia contra la que se miden todos
los demás. Dice cuánta variación territorial hay **antes** de descontar nada, y por tanto
cuánta puede llegar a explicarse.

**Las dos medidas que salen de él.**

**ICC**, coeficiente de correlación intraclase: la proporción de la variación total del
riesgo latente que ocurre entre territorios. Se calcula por el método de la variable latente,
que fija la varianza individual en $\pi^2/3$ —la varianza de la logística estándar— porque en
un modelo con desenlace binario esa varianza no está identificada. Es la convención en
epidemiología social.

**MOR**, razón de momios mediana. Es la que hace interpretable el resultado. La varianza de
los efectos aleatorios está en escala logarítmica y no se puede comparar con las razones de
momios de los efectos fijos; la MOR la traduce a la misma escala:

$$\text{MOR}=\exp\left(\sqrt{2\sigma^2}\cdot\Phi^{-1}(0{,}75)\right)$$

Es el valor mediano de la razón de momios al comparar dos personas idénticas de dos
territorios distintos, tomando siempre el de mayor riesgo como numerador. Una MOR de 1,30
significa que mover a una persona a otro municipio cambiaría su momio en un 30 % en mediana.
**Eso sí se puede comparar con el efecto de un determinante social**, y esa comparación es lo
que da sentido al modelo.

In [3]:
# ---------------------------------------------------------------------------
# M0: modelo vacio
# ---------------------------------------------------------------------------

# agregar() reduce la muestra a celdas binomiales. Sin covariables, la celda es
# el municipio: 1.157 filas en vez de diecisiete millones.
celdas_m0 = multinivel.agregar(muestra, [], "MUNI_RES", "DPTO_RES")

# Sin covariables, la matriz de disenio es una sola columna de unos.
X0 = np.ones((len(celdas_m0), 1))

with cronometro("M0"):
    MODELOS["M0 vacio"] = multinivel.ajustar(
        celdas_m0, X0, "MUNI_RES", "DPTO_RES", ["intercepto"])

m0 = MODELOS["M0 vacio"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

medidas = multinivel.icc(m0)
print("\nCOMPONENTES DE VARIANZA\n")
print(f"  sigma municipio      {m0['sigma_grupo']:.4f}   "
      f"varianza {medidas['varianza_grupo']:.4f}")
print(f"  sigma departamento   {m0['sigma_supergrupo']:.4f}   "
      f"varianza {medidas['varianza_supergrupo']:.4f}")
print(f"  varianza individual  {medidas['varianza_individual']:.4f}   "
      f"(pi^2/3, por construccion)")
print()
print(f"  ICC departamento     {medidas['icc_supergrupo']*100:6.2f} %   "
      "dos nacimientos de municipios distintos del mismo departamento")
print(f"  ICC municipio        {medidas['icc_grupo']*100:6.2f} %   "
      "dos nacimientos del mismo municipio")
print()
print(f"  MOR municipio        {multinivel.mor(m0['sigma_grupo']):.3f}")
print(f"  MOR departamento     {multinivel.mor(m0['sigma_supergrupo']):.3f}")

# La prevalencia que implica el intercepto, para comprobar que el modelo esta
# anclado donde debe. POR QUE NO COINCIDE EXACTAMENTE CON LA PREVALENCIA
# OBSERVADA: el intercepto de un modelo logistico con efectos aleatorios es la
# probabilidad del municipio MEDIANO, no la media de la poblacion. La
# diferencia entre las dos es un efecto conocido de la no linealidad de la
# funcion logistica, y verla es una comprobacion de que el modelo esta bien
# especificado, no un problema.
from scipy.special import expit
print(f"\n  prevalencia del municipio mediano  {expit(m0['beta'][0])*100:.3f} %")
print(f"  prevalencia observada global       "
      f"{m0['n_casos']/m0['n_nacimientos']*100:.3f} %")

  filas originales       17,376,860
  descartadas                     0  (0.00 %)
  celdas binomiales           1,157  (reduccion 15019x)
  nacimientos por celda: mediana 3871, maximo 2,717,133
[inicio] M0
  ajustando modelo de 3 niveles: 1 efectos fijos, 1157 MUNI_RES, 33 DPTO_RES
  convergio en 15 iteraciones, loglik = -6,354.1
[fin]    M0  ->  0.9 s

COMPONENTES DE VARIANZA

  sigma municipio      0.1772   varianza 0.0314
  sigma departamento   0.1693   varianza 0.0287
  varianza individual  3.2899   (pi^2/3, por construccion)

  ICC departamento       0.86 %   dos nacimientos de municipios distintos del mismo departamento
  ICC municipio          1.79 %   dos nacimientos del mismo municipio

  MOR municipio        1.184
  MOR departamento     1.175

  prevalencia del municipio mediano  7.255 %
  prevalencia observada global       8.780 %


**Cómo se lee esta salida.**

El resultado sobre los 17.376.860 nacimientos, 1.157 municipios y 33 departamentos:

| | Valor |
|---|---:|
| σ municipio | 0,1772 |
| σ departamento | 0,1693 |
| **ICC municipio** | **1,79 %** |
| ICC departamento | 0,86 % |
| **MOR municipio** | **1,184** |
| MOR departamento | 1,175 |

**El ICC municipal es del 1,79 %**, y hay que saber presentarlo. Suena pequeño, y comparado
con la variación individual lo es: el 98 % de la variación en la propensión al bajo peso al
nacer ocurre entre personas, no entre municipios. Eso es lo esperable en un desenlace de
salud individual, y un ICC del 20 % sería la señal de que algo está mal especificado.

**Lo que no se puede decir es que el territorio «no importa».** La MOR municipal de 1,184
dice que mover a una mujer idéntica a otro municipio cambiaría su momio de bajo peso al nacer
en un **18 % en mediana**. Puesto sobre 17 millones de nacimientos, eso es un efecto
poblacional grande. La MOR existe precisamente porque el ICC induce a subestimar la
importancia del contexto.

**El departamento aporta casi tanto como el municipio**, con σ de 0,169 frente a 0,177. Eso
significa que buena parte de la heterogeneidad territorial opera a escala departamental y no
municipal: municipios del mismo departamento se parecen bastante entre sí. Tiene sentido en
un país donde la organización del sistema de salud es departamental.

**La prevalencia del municipio mediano es menor que la global**, y no es un error: el
intercepto de un modelo logístico con efectos aleatorios corresponde al municipio mediano, no
al promedio poblacional. La diferencia viene de la no linealidad de la función logística y su
presencia confirma que el modelo está bien especificado.

**Correspondencia con el manuscrito.** ICC y MOR de M0 al Cuadro 5.4, primera columna.

## 2. Modelo 1: el efecto total de la posición social

**Qué hacemos.** Añadimos los cuatro determinantes sociales: nivel educativo materno, estado
conyugal, régimen de afiliación y área de residencia.

**Qué estima este modelo, dicho con precisión.** El **efecto total** de la posición social
sobre el bajo peso al nacer, descontada la confusión territorial. «Total» quiere decir que
incluye todo lo que ocurre por consecuencia de la posición social: si tener menos educación
lleva a menos controles prenatales y eso a su vez aumenta el riesgo, ese camino está dentro
del coeficiente.

**Por qué NO se incluyen aquí las variables obstétricas.** Porque son consecuencia de la
posición social, no causa. Incluirlas ahora respondería a otra pregunta, y es el asunto del
bloque 4.

**La referencia de cada variable es su primera categoría**, que en las escalas ordinales del
proyecto es el extremo más desfavorecido. Todas las razones de momios se leen contra ella:
cada coeficiente dice cuánto cambia el riesgo al subir desde el escalón más bajo.

**Lo que el modelo controla y lo que no.** Controla el municipio y el departamento por efectos
aleatorios, de modo que las comparaciones entre niveles educativos son **dentro** del mismo
territorio. Eso es exactamente lo que las prevalencias crudas de la Fase 1 y el índice de
concentración de la Fase 3 **no** hacen.

In [4]:
# ---------------------------------------------------------------------------
# M1: determinantes sociales
# ---------------------------------------------------------------------------

SOCIALES = ["EDUC4_MADRE", "ESTCIV5", "SEG4", "AREA_RES"]

celdas_m1 = multinivel.agregar(muestra, SOCIALES, "MUNI_RES", "DPTO_RES")
X1, nombres_m1 = multinivel.matriz_diseno(celdas_m1, SOCIALES, ETIQUETAS)

print(f"\n  matriz de disenio: {X1.shape[0]:,} celdas x {X1.shape[1]} columnas")
print("  categorias de referencia:")
for var, ref in multinivel.referencias(celdas_m1, SOCIALES, ETIQUETAS).items():
    print(f"     {var:<14} {ref}")

with cronometro("M1"):
    MODELOS["M1 social"] = multinivel.ajustar(
        celdas_m1, X1, "MUNI_RES", "DPTO_RES", nombres_m1)

m1 = MODELOS["M1 social"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

tabla_m1 = multinivel.tabla_efectos(m1)
guardar_tabla(tabla_m1, "multinivel_m1_social", decimales=4)

print("\nEFECTOS FIJOS — razones de momios contra la categoria de referencia\n")
print(tabla_m1.to_string(index=False))

print(f"\n  sigma municipio     {m1['sigma_grupo']:.4f}  "
      f"(M0: {m0['sigma_grupo']:.4f})")
print(f"  sigma departamento  {m1['sigma_supergrupo']:.4f}  "
      f"(M0: {m0['sigma_supergrupo']:.4f})")
print(f"  PCV municipal       {multinivel.pcv(m0, m1, 'grupo')*100:+.1f} %")
print(f"  PCV departamental   {multinivel.pcv(m0, m1, 'supergrupo')*100:+.1f} %")

  filas originales       17,376,860
  descartadas             1,235,196  (7.11 %)
  celdas binomiales         124,635  (reduccion 130x)
  nacimientos por celda: mediana 6, maximo 564,154

  matriz de disenio: 124,635 celdas x 13 columnas
  categorias de referencia:
     EDUC4_MADRE    Ninguno o preescolar
     ESTCIV5        Casada
     SEG4           Contributivo
     AREA_RES       Cabecera
[inicio] M1
  ajustando modelo de 3 niveles: 13 efectos fijos, 1157 MUNI_RES, 33 DPTO_RES
  convergio en 123 iteraciones, loglik = -156,457.9
[fin]    M1  ->  151.6 min
tabla guardada: multinivel_m1_social.csv y multinivel_m1_social.tex

EFECTOS FIJOS — razones de momios contra la categoria de referencia

                                        termino  coeficiente  razon_momios
                                     intercepto      -2.5658        0.0769
  EDUC4_MADRE: Primaria vs Ninguno o preescolar      -0.1572        0.8545
EDUC4_MADRE: Secundaria vs Ninguno o preescolar      -0.1566        0.85

**Cómo se lee esta salida, y contiene el resultado más importante de la fase.**

Las razones de momios obtenidas, todas contra la categoría más desfavorecida de cada variable:

| Término | RM |
|---|---:|
| **`EDUC4_MADRE` = Primaria** | **0,855** |
| **`EDUC4_MADRE` = Secundaria** | **0,855** |
| **`EDUC4_MADRE` = Superior** | **0,872** |
| `ESTCIV5` = Unión libre | 1,098 |
| **`ESTCIV5` = Soltera** | **1,287** |
| `ESTCIV5` = Separada o divorciada | 1,147 |
| `ESTCIV5` = Viuda | 1,079 |
| `SEG4` = Subsidiado | 1,046 |
| `SEG4` = No asegurado | 0,967 |
| `AREA_RES` = Rural disperso | 1,098 |

**El gradiente educativo se da la vuelta al controlar el territorio.** Las tres categorías
educativas tienen razón de momios **menor que 1** frente a «ninguno o preescolar»: cualquier
nivel de educación se asocia a un 13-15 % menos de momio de bajo peso al nacer que no tener
ninguno. Es el **gradiente social clásico**, y es lo contrario de lo que dieron la prevalencia
cruda de la Fase 1 y el índice de concentración de la Fase 3.

**Por qué cambia el signo, y es un resultado, no una contradicción.** Las dos fases
anteriores comparan madres de distinta educación **sin importar dónde viven**. Las madres sin
educación se concentran en municipios donde el bajo peso se detecta y registra menos, de modo
que su prevalencia registrada sale artificialmente baja. Al controlar el municipio, la
comparación pasa a ser **entre madres del mismo municipio**, y ahí el gradiente aparece.

Esto es confusión territorial en estado puro, y es la justificación empírica de por qué la
tesis necesita el modelo multinivel: **no es un refinamiento del análisis anterior, es lo que
corrige su sesgo**.

**Pero el gradiente no tiene escalones.** Las tres razones de momios son prácticamente
iguales —0,855, 0,855, 0,872—, de modo que lo que el modelo detecta es un contraste entre
**tener educación o no tenerla**, no una relación dosis-respuesta. Conviene decirlo así y no
hablar de «gradiente educativo» sin matizar.

**El determinante social con más efecto no es la educación: es el estado conyugal.** Ser
soltera se asocia a una razón de momios de **1,287** frente a estar casada, el efecto más
grande de todo el modelo y mayor que la MOR municipal de 1,184. Es un resultado con
literatura detrás —apoyo social y económico durante el embarazo— y merece un párrafo propio
en el Capítulo 5.

**El régimen de afiliación casi no aporta** una vez controlado el territorio, con razones de
momios entre 0,97 y 1,05. Eso contrasta con la Fase 3, donde el aseguramiento daba el
gradiente más marcado de los dos ordenamientos, y la explicación es la misma: buena parte de
lo que parecía efecto del régimen era efecto del lugar.

**El PCV municipal es negativo, −6,1 %.** No es un error: añadir los determinantes sociales
**aumentó** la varianza entre municipios. Ocurre cuando la composición social de los
municipios enmascaraba parte de la heterogeneidad territorial; al controlarla, las
diferencias entre municipios se hacen más visibles. Hay que reportarlo como lo que es.

**Correspondencia con el manuscrito.** La tabla de efectos al Cuadro 5.5; el cambio de signo
del gradiente educativo respecto de las Fases 1 y 3, a la Sección 5.3.2 y a la discusión del
Capítulo 6. Es el hallazgo que articula el capítulo.

## 3. Modelo 2: el efecto independiente, ajustado por edad, paridad y multiplicidad

**Este es el modelo que responde al tercer objetivo específico de la tesis**, que pide
estimar el efecto independiente de cada determinante social «ajustados por edad materna,
paridad y multiplicidad del embarazo».

**Qué hacemos.** Añadimos las tres covariables obstétricas que el objetivo nombra.

**Por qué las tres son confusores y no mediadores**, que es lo que justifica meterlas aquí y
no en el modelo siguiente:

**Edad materna.** Influye en el peso al nacer por vías biológicas —maduración uterina en la
adolescencia, comorbilidad en la edad avanzada— y al mismo tiempo se distribuye de forma muy
desigual entre niveles educativos: las madres adolescentes tienen, por construcción, menos
educación completada. Sin controlarla, el coeficiente educativo recoge parte del efecto de la
edad.

**Paridad.** El número de hijos previos afecta al riesgo obstétrico —la primiparidad se
asocia a menor peso— y correlaciona con la posición socioeconómica.

**Multiplicidad del embarazo.** Es el determinante proximal más fuerte del bajo peso después
de la edad gestacional: la V de Cramér contra `BPN` fue de **0,242** en la Fase 1, la segunda
más alta de las quince variables. El **1,81 %** de los nacimientos son múltiples y concentran
el **12,16 %** de todos los casos de bajo peso, con una prevalencia de **59,21 %** frente al
**7,87 %** de los partos simples — **siete veces y media mayor**.

**Qué papel juega exactamente, comprobado y no supuesto.** La objeción evidente es que la
multiplicidad podría ser un **mediador**: el acceso a tratamientos de reproducción asistida
aumenta la probabilidad de embarazo múltiple y depende de la capacidad de pago, de modo que
habría una vía de la posición social a la multiplicidad. Es plausible en teoría, y se
comprobó:

| | % de partos múltiples |
|---|---:|
| Sin educación | 2,305 % |
| Primaria | 1,950 % |
| Secundaria | 1,640 % |
| Superior | 2,061 % |

V de Cramér de la multiplicidad contra **educación materna: 0,0106**; contra **aseguramiento:
0,0070**. Frente al **0,2419** contra el desenlace.

**La vía no existe en estos datos.** La multiplicidad se asocia fuertísimamente con el bajo
peso y **prácticamente nada** con la posición social, de modo que no es mediador. Y tampoco es
un confusor en sentido estricto: un confusor tiene que asociarse con la exposición **y** con
el desenlace, y con la exposición se asocia 0,01.

**Lo que es, con el nombre técnico correcto, es una covariable de precisión.** Un predictor
biológico fuerte e independiente de la posición social. Ajustar por ella no elimina confusión
—apenas hay que eliminar— sino que **reduce la varianza residual y mejora la precisión de los
coeficientes sociales**. Esa es la justificación que va al manuscrito, y es más defendible
que apelar a la costumbre obstétrica.

**La pequeña no monotonía** —2,305 % en el extremo bajo y 2,061 % en el alto, con el mínimo en
secundaria— es compatible con dos mecanismos opuestos: reproducción asistida en el extremo
alto, y mayor edad materna y paridad en el bajo. Con una V de 0,011 ninguno de los dos mueve
los coeficientes sociales de forma apreciable.

**Las referencias de estas tres no son la primera categoría.** En los determinantes sociales
la referencia es el extremo más desfavorecido, porque interesa leer el gradiente hacia
arriba. En las covariables de ajuste la referencia es el **grupo de menor riesgo**, que es
como se reporta en obstetricia: de 20 a 34 años, un hijo, y parto simple.

In [ ]:
# ---------------------------------------------------------------------------
# M2: + covariables obstetricas de ajuste  (objetivo especifico 3)
# ---------------------------------------------------------------------------

# --- Edad materna ----------------------------------------------------------
# Viene en nueve bandas quinquenales, de 10-14 a 50-54, identicas en los dos
# regimenes de codificacion. Se agrupan en los tres tramos obstetricos
# estandar.
#
# POR QUE AGRUPAR: con nueve bandas, las de los extremos tienen muy pocos
# registros —3.379 nacimientos en la de 50 a 54 anios— y producirian
# coeficientes inestables. Los tres tramos son los que usa la literatura y los
# que tienen interpretacion clinica.
edad = pd.to_numeric(muestra["EDAD_MADRE"], errors="coerce")
muestra["EDAD3"] = pd.cut(
    edad, bins=[0.5, 2.5, 5.5, 9.5],
    labels=["20 a 34", "menor de 20", "35 o mas"])
# OJO con el orden de las etiquetas: pd.cut las asigna por orden de los
# intervalos, de modo que la etiqueta "20 a 34" corresponde a los codigos 1 y
# 2, que son 10-14 y 15-19. Se corrige asignando explicitamente.
muestra["EDAD3"] = pd.Categorical(
    np.select(
        [edad <= 2, (edad >= 3) & (edad <= 5), edad >= 6],
        ["menor de 20", "20 a 34", "35 o mas"], default=None),
    categories=["20 a 34", "menor de 20", "35 o mas"])

# --- Paridad ---------------------------------------------------------------
# N_HIJOSV es el numero de hijos nacidos vivos, incluido el actual: el 43,8 %
# de los registros vale 1, es decir primiparas. Se agrupa en cuatro, porque
# por encima de cuatro hijos los efectivos caen rapidamente.
hijos = pd.to_numeric(muestra["N_HIJOSV"], errors="coerce")
muestra["PARIDAD4"] = pd.Categorical(
    np.select([hijos == 1, hijos == 2, hijos == 3, hijos >= 4],
              ["1 hijo", "2 hijos", "3 hijos", "4 o mas"], default=None),
    categories=["1 hijo", "2 hijos", "3 hijos", "4 o mas"])

# --- Multiplicidad ---------------------------------------------------------
# Triple y cuadruple suman 7.520 nacimientos en toda la serie; separarlos
# daria coeficientes con intervalos enormes. Se unen en "triple o mas".
mult = pd.to_numeric(muestra["MUL_PARTO"], errors="coerce")
muestra["MULTIPLE3"] = pd.Categorical(
    np.select([mult == 1, mult == 2, mult >= 3],
              ["simple", "doble", "triple o mas"], default=None),
    categories=["simple", "doble", "triple o mas"])

print("COVARIABLES DE AJUSTE\n")
for var in ["EDAD3", "PARIDAD4", "MULTIPLE3"]:
    print(f"{var}   ({muestra[var].notna().mean()*100:.2f} % informado)")
    for k, n in muestra[var].value_counts(dropna=False).items():
        print(f"   {str(k):<14} {n:>12,}  {n/len(muestra)*100:5.2f} %")
    print()

# --- Que papel juega la multiplicidad: comprobado, no supuesto -------------
#
# POR QUE ESTE BLOQUE EXISTE: la objecion evidente a tratar la multiplicidad
# como covariable de ajuste es que podria ser un MEDIADOR, porque el acceso a
# reproduccion asistida depende de la capacidad de pago. Es una hipotesis
# plausible y verificable, y verificarla cuesta dos tablas.
#
# La regla: un mediador tiene que DEPENDER de la exposicion. Si la
# multiplicidad no se asocia con la posicion social, no hay via que mediar.
import asociacion

print("=" * 66)
print("QUE PAPEL JUEGA LA MULTIPLICIDAD\n")

cruce = (muestra.groupby("MULTIPLE3", observed=True)["BPN"]
                .agg(n="size", casos="sum"))
cruce["prevalencia_pct"] = (cruce["casos"] / cruce["n"] * 100).round(2)
cruce["pct_poblacion"] = (cruce["n"] / len(muestra) * 100).round(3)
guardar_tabla(cruce.reset_index(), "multiplicidad_por_bpn", decimales=3)
print("Multiplicidad contra el desenlace:\n")
print(cruce.to_string())

multiples = cruce.loc[cruce.index != "simple"]
n_mult, casos_mult = int(multiples["n"].sum()), int(multiples["casos"].sum())
prev_simple = cruce.loc["simple", "prevalencia_pct"]
print(f"\n  multiples: {n_mult:,} ({n_mult/len(muestra)*100:.2f} % de la poblacion)")
print(f"  prevalencia simple {prev_simple:.2f} % contra multiple "
      f"{casos_mult/n_mult*100:.2f} %  ->  "
      f"{(casos_mult/n_mult*100)/prev_simple:.1f} veces mayor")
print(f"  concentran el {casos_mult/cruce['casos'].sum()*100:.2f} % de todos "
      "los casos de bajo peso")

print("\nMultiplicidad contra la posicion social:\n")
for var in ["EDUC4_MADRE", "SEG4"]:
    reparto = pd.crosstab(muestra[var], muestra["MULTIPLE3"],
                          normalize="index").mul(100)
    pct_mult = reparto.drop(columns=["simple"]).sum(axis=1)
    etqs = ETIQUETAS.get(var, {})
    print(f"  {var}")
    for nivel, valor in pct_mult.items():
        print(f"     {str(etqs.get(nivel, nivel)):<40} {valor:6.3f} % multiples")
    v = asociacion.v_cramer(muestra[var], muestra["MULTIPLE3"])
    print(f"     V de Cramer = {v:.4f}\n")

print("  referencia: V de Cramer multiplicidad x BPN = 0,2419 (Fase 1)")
print("\n  Asociacion fuerte con el desenlace y practicamente nula con la")
print("  posicion social: no hay via que mediar. Es una covariable de")
print("  PRECISION, no un mediador ni un confusor en sentido estricto.")
print("=" * 66 + "\n")

AJUSTES = ["EDAD3", "PARIDAD4", "MULTIPLE3"]
SOCIALES_AJUSTADO = SOCIALES + AJUSTES

# La referencia de cada covariable de ajuste es el grupo de MENOR RIESGO, no
# la primera categoria alfabetica. Es como se reporta en obstetricia.
REFERENCIAS_FIJAS = {"EDAD3": "20 a 34", "PARIDAD4": "1 hijo",
                     "MULTIPLE3": "simple"}
for v in AJUSTES:
    ETIQUETAS[v] = {}

celdas_m2 = multinivel.agregar(muestra, SOCIALES_AJUSTADO, "MUNI_RES", "DPTO_RES")
X2, nombres_m2 = multinivel.matriz_diseno(celdas_m2, SOCIALES_AJUSTADO,
                                          ETIQUETAS, REFERENCIAS_FIJAS)

print("categorias de referencia:")
for var, ref in multinivel.referencias(celdas_m2, SOCIALES_AJUSTADO,
                                       ETIQUETAS, REFERENCIAS_FIJAS).items():
    print(f"   {var:<14} {ref}")

with cronometro("M2"):
    MODELOS["M2 ajustado"] = multinivel.ajustar(
        celdas_m2, X2, "MUNI_RES", "DPTO_RES", nombres_m2)

m2 = MODELOS["M2 ajustado"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

guardar_tabla(multinivel.tabla_efectos(m2), "multinivel_m2_ajustado", decimales=4)
print("\nEFECTOS FIJOS\n")
print(multinivel.tabla_efectos(m2).to_string(index=False))

# --- Cuanto cambian los coeficientes sociales al ajustar -------------------
#
# Se compara contra M2 reajustado sobre la MISMA submuestra. POR QUE: las tres
# covariables tienen nulos, de modo que M3 usa menos filas que M2 y la
# diferencia entre coeficientes mezclaria el efecto del ajuste con el del
# cambio de poblacion.
sub_m2 = muestra.dropna(subset=AJUSTES)
print(f"\nsubmuestra comun: {len(sub_m2):,} de {len(muestra):,} "
      f"({len(sub_m2)/len(muestra)*100:.1f} %)")

celdas_m1c = multinivel.agregar(sub_m2, SOCIALES, "MUNI_RES", "DPTO_RES",
                                verbose=False)
X1c, nombres_m1c = multinivel.matriz_diseno(celdas_m1c, SOCIALES, ETIQUETAS)
with cronometro("M1 sobre la submuestra de M2"):
    m1c = multinivel.ajustar(celdas_m1c, X1c, "MUNI_RES", "DPTO_RES",
                             nombres_m1c, verbose=False)
MODELOS["M1c (submuestra de M2)"] = m1c
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

comun = [n for n in m1c["nombres"] if n in m2["nombres"] and n != "intercepto"]
ajuste = pd.DataFrame({
    "termino": comun,
    "rm_sin_ajuste": [np.exp(m1c["beta"][m1c["nombres"].index(n)]) for n in comun],
    "rm_ajustado": [np.exp(m2["beta"][m2["nombres"].index(n)]) for n in comun],
})
ajuste["cambio_pct"] = ((ajuste["rm_ajustado"] / ajuste["rm_sin_ajuste"] - 1)
                        * 100).round(2)
guardar_tabla(ajuste, "multinivel_efecto_del_ajuste", decimales=4)

print("\nEFECTO DE AJUSTAR POR EDAD, PARIDAD Y MULTIPLICIDAD\n")
print(ajuste.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

print(f"\n  sigma municipio  M1c {m1c['sigma_grupo']:.4f} -> M2 {m2['sigma_grupo']:.4f}")
print(f"  PCV municipal por el ajuste: "
      f"{multinivel.pcv(m1c, m2, 'grupo')*100:+.1f} %")

COVARIABLES DE AJUSTE

EDAD3   (99.80 % informado)
   20 a 34          11,794,738  67.88 %
   menor de 20       3,717,626  21.39 %
   35 o mas          1,829,082  10.53 %
   nan                  35,414   0.20 %

PARIDAD4   (99.27 % informado)
   1 hijo            7,615,450  43.83 %
   2 hijos           5,298,773  30.49 %
   3 hijos           2,473,760  14.24 %
   4 o mas           1,862,645  10.72 %
   nan                 126,232   0.73 %

MULTIPLE3   (99.31 % informado)
   simple           16,945,108  97.52 %
   doble               304,200   1.75 %
   nan                 120,032   0.69 %
   triple o mas          7,520   0.04 %

QUE PAPEL JUEGA LA MULTIPLICIDAD

tabla guardada: multiplicidad_por_bpn.csv y multiplicidad_por_bpn.tex
Multiplicidad contra el desenlace:

                     n    casos  prevalencia_pct  pct_poblacion
MULTIPLE3                                                      
simple        16945108  1332886             7.87         97.515
doble           304200   179924

**Cómo se lee esta salida.**

**Este es el cuadro que responde al objetivo específico 3**, y es el que va al Capítulo 5 como
estimación del efecto independiente de cada determinante social.

**Las tres covariables de ajuste deben comportarse como predice la obstetricia**, y eso es la
primera comprobación de que el modelo está bien:

- **Edad materna**: tanto «menor de 20» como «35 o más» deben salir **por encima de 1** frente
  al tramo de 20 a 34. Es la relación en U clásica. Si alguno saliera por debajo, habría que
  revisar la construcción de la variable.
- **Paridad**: los coeficientes se leen contra las primíparas. La primiparidad se asocia a
  menor peso, de modo que las categorías de 2 y 3 hijos deben salir **por debajo de 1**.
- **Multiplicidad**: «doble» y «triple o más» deben salir muy por encima de 1, y con
  diferencia el efecto más grande de todo el modelo. Un embarazo múltiple casi garantiza bajo
  peso al nacer.

**La columna `cambio_pct` es lo que hay que interpretar.** Dice cuánto se mueve cada razón de
momios social al descontar las tres covariables:

- **Cambios pequeños, por debajo del 5 %**: la edad, la paridad y la multiplicidad no estaban
  confundiendo apreciablemente ese efecto social. El coeficiente de M2 ya era casi
  independiente.
- **Cambios grandes en educación**: parte de lo que M2 atribuía al nivel educativo era en
  realidad efecto de la edad materna, porque las madres con menos educación son más jóvenes.
  Es el caso que más conviene vigilar, y si ocurre hay que decirlo: **la brecha educativa
  bruta exagera la brecha educativa independiente.**

**Sobre la pérdida de muestra.** Las tres covariables tienen nulos, de modo que este modelo
se ajusta sobre el 99 % de la muestra y no sobre el 100 %. La celda la reporta y
compara solo contra un M1 reajustado sobre exactamente las mismas filas. Sin esa precaución,
la diferencia entre coeficientes mezclaría el efecto del ajuste con el de cambiar de
población, y no se sabría cuál produjo qué.

**Correspondencia con el manuscrito.** La tabla de efectos de M3 al Cuadro 5.5 como columna
principal; la tabla de cambio al Cuadro 5.6. En el texto del Capítulo 5, la frase que responde
al objetivo 3 se construye con **estas** razones de momios, no con las de M1.

**Por qué la etnia no está en este modelo.** `IDPERTET` no existe antes de 2008, de modo que
incluirla aquí costaría el 43 % de la muestra. El objetivo 3 no la nombra entre los ajustes,
así que entra en el modelo siguiente, sobre su propia submuestra y declarada como tal.

## 4. Modelo 3: descontando la composición étnica

**Qué hacemos.** Añadimos el reconocimiento étnico del recién nacido.

**Por qué en un paso propio y el último de los confusores.** Primero, porque `IDPERTET` no
existe antes de 2008 y su inclusión cuesta el 43 % de la muestra: aislarla en su propio
modelo permite que M2, que responde al objetivo 3, se ajuste sobre el 99 % de los datos.
Segundo, Por lo que mostró la
matriz de asociación de la Fase 1: `IDPERTET` tiene V de Cramér de **0,473** con el municipio
y **0,378** con el departamento. La etnia y el territorio se solapan fuertemente en Colombia,
porque los pueblos indígenas y las comunidades afrodescendientes se concentran en
departamentos concretos.

Eso tiene una consecuencia que hay que anticipar: **el modelo multinivel ya está absorbiendo
buena parte del efecto étnico en los interceptos aleatorios de municipio**. Un coeficiente
étnico pequeño en este modelo no significa que la etnia no importe; significa que lo que
aporta por encima del municipio de residencia es poco, que es una afirmación distinta.

**La limitación que hay que declarar.** `IDPERTET` no existe antes de 2008 —es ausencia
estructural, no dato faltante—, de modo que este modelo se ajusta sobre el 60 % de la muestra
en que la variable existe. Los coeficientes de M3 **no son directamente comparables** con los
de M2, que usa toda la serie. La comparación honesta exige reajustar M2 sobre la misma
submuestra, y eso es lo que hace la celda.

In [ ]:
# ---------------------------------------------------------------------------
# M3: + reconocimiento etnico
# ---------------------------------------------------------------------------

SOCIALES_ETNIA = SOCIALES_AJUSTADO + ["IDPERTET"]

celdas_m3 = multinivel.agregar(muestra, SOCIALES_ETNIA, "MUNI_RES", "DPTO_RES")
X3, nombres_m3 = multinivel.matriz_diseno(celdas_m3, SOCIALES_ETNIA,
                                          ETIQUETAS, REFERENCIAS_FIJAS)

with cronometro("M3"):
    MODELOS["M3 +etnia"] = multinivel.ajustar(
        celdas_m3, X3, "MUNI_RES", "DPTO_RES", nombres_m3)

m3 = MODELOS["M3 +etnia"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

guardar_tabla(multinivel.tabla_efectos(m3), "multinivel_m3_etnia", decimales=4)
print("\nEFECTOS FIJOS\n")
print(multinivel.tabla_efectos(m3).to_string(index=False))

# --- M1 reajustado sobre la MISMA submuestra --------------------------------
#
# POR QUE HACE FALTA: M2 solo usa los anios en que existe IDPERTET, que son el
# 60 % de la muestra. Comparar sus coeficientes con los de M1, ajustado sobre
# los 27 anios, mezclaria el efecto de anadir la etnia con el de cambiar de
# poblacion. La unica comparacion interpretable es contra un M1 ajustado sobre
# exactamente las mismas filas.
print("\n" + "="*70)
print("M2 REAJUSTADO SOBRE LA SUBMUESTRA DE M3, para que la comparacion valga")
submuestra = muestra.dropna(subset=["IDPERTET"] + AJUSTES)
print(f"  submuestra: {len(submuestra):,} de {len(muestra):,} "
      f"({len(submuestra)/len(muestra)*100:.1f} %)")

celdas_m2b = multinivel.agregar(submuestra, SOCIALES_AJUSTADO, "MUNI_RES",
                                "DPTO_RES", verbose=False)
X2b, nombres_m2b = multinivel.matriz_diseno(celdas_m2b, SOCIALES_AJUSTADO,
                                            ETIQUETAS, REFERENCIAS_FIJAS)
with cronometro("M2 sobre submuestra"):
    MODELOS["M2b (submuestra de M3)"] = multinivel.ajustar(
        celdas_m2b, X2b, "MUNI_RES", "DPTO_RES", nombres_m2b, verbose=False)
m2b = MODELOS["M2b (submuestra de M3)"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

# --- Cuanto se mueven los coeficientes sociales al anadir la etnia ---------
comun = [n for n in m2b["nombres"] if n in m3["nombres"]]
cambio = pd.DataFrame({
    "termino": comun,
    "rm_sin_etnia": [np.exp(m2b["beta"][m2b["nombres"].index(n)]) for n in comun],
    "rm_con_etnia": [np.exp(m3["beta"][m3["nombres"].index(n)]) for n in comun],
})
cambio["cambio_pct"] = ((cambio["rm_con_etnia"] / cambio["rm_sin_etnia"] - 1)
                        * 100).round(2)
guardar_tabla(cambio, "multinivel_cambio_por_etnia", decimales=4)

print("\nCUANTO CAMBIAN LOS COEFICIENTES SOCIALES AL ANADIR LA ETNIA\n")
print(cambio.to_string(index=False, float_format=lambda x: f"{x:8.4f}"))

print(f"\n  sigma municipio  M2b {m2b['sigma_grupo']:.4f} -> M3 {m3['sigma_grupo']:.4f}")
print(f"  PCV municipal por anadir la etnia: "
      f"{multinivel.pcv(m2b, m3, 'grupo')*100:+.1f} %")

NameError: name 'SOCIALES_AJUSTADO' is not defined

**Cómo se lee esta salida.**

**Primero, la comparación justa.** Los coeficientes de M2 solo se comparan contra el M1
reajustado sobre la misma submuestra. Comparar contra el M2 de los 27 años mezclaría dos
cambios —añadir la etnia y cambiar de población— y no se sabría cuál produjo qué.

**Qué mirar en la columna `cambio_pct`.** Es cuánto se mueve cada razón de momios social al
descontar la composición étnica:

- **Cambios por debajo del 5 %**: la etnia no confundía apreciablemente ese efecto social.
- **Cambios grandes en educación o aseguramiento**: parte de lo que M2 atribuía a la posición
  socioeconómica era en realidad composición étnica, y hay que declararlo.

**Lo que hay que anticipar sobre los coeficientes étnicos.** Es probable que salgan pequeños,
y la lectura correcta **no** es «la etnia no importa». El modelo ya lleva interceptos
aleatorios de municipio, y con V de Cramér de 0,473 entre etnia y municipio, esos interceptos
absorben la mayor parte del efecto étnico antes de que el coeficiente fijo lo vea. Lo que el
coeficiente mide es **lo que la etnia aporta por encima del lugar de residencia**, que es una
cantidad distinta y mucho menor.

**Esa es la limitación estructural del diseño para el eje étnico**, y conviene declararla en
lugar de dejar que el jurado la encuentre: con estos datos no se puede separar limpiamente
«ser indígena» de «vivir donde vive la población indígena». La alternativa —quitar los
interceptos municipales para que el efecto étnico aflore— produciría un coeficiente inflado
por confusión territorial, y sería peor.

**Correspondencia con el manuscrito.** La tabla de cambio de coeficientes al Cuadro 5.6; la
discusión sobre la inseparabilidad de etnia y territorio, a las limitaciones del Capítulo 6.

## 5. Modelo 4: confusor y mediador, y por qué el ajuste cambia la pregunta

**Este bloque contiene la decisión metodológica que un jurado va a examinar.** Conviene
leerlo entero antes de mirar los números.

**Qué hacemos.** Añadimos el número de consultas prenatales.

**Por qué no está entre las covariables de ajuste del bloque anterior.** Porque no es un
confusor: es un **mediador**. Y la diferencia entre los dos no es de grado, es de naturaleza.

**Un confusor** es una causa común del determinante y del desenlace. El municipio de
residencia es un confusor de la relación entre educación y bajo peso al nacer: influye en la
educación a la que se accede y, por separado, en el riesgo obstétrico. Si no se controla, el
coeficiente de educación recoge parte del efecto del lugar y está **sesgado**. Controlarlo
**corrige** la estimación.

**Un mediador** está en el camino causal, entre el determinante y el desenlace. El número de
consultas prenatales es un mediador: tener menos educación lleva a menos controles, y menos
controles aumentan el riesgo. Controlarlo **no corrige** nada. Cambia la cantidad que se
estima.

$$\text{educación}\;\longrightarrow\;\text{consultas prenatales}\;\longrightarrow\;\text{bajo peso}$$

$$\text{educación}\;\longrightarrow\;\text{bajo peso}\quad(\text{efecto directo})$$

**Las dos cantidades y sus dos preguntas.**

| | Qué estima | A qué pregunta responde |
|---|---|---|
| **Sin el mediador** (M3) | **efecto total**, ya ajustado por los confusores | ¿cuánto bajo peso al nacer se evitaría si todas las madres tuvieran educación superior? |
| **Con el mediador** (M4) | **efecto directo** | ¿cuánto quedaría si además se igualara el acceso a atención prenatal? |

**Ninguna de las dos está «mejor controlada» que la otra.** Son respuestas a preguntas
distintas, y la de política pública es la primera: al diseñar una intervención educativa,
interesa el efecto total, incluido todo lo que ocurre por su consecuencia.

**Por eso el coeficiente de educación DEBE atenuarse en M4 respecto de M3, y esa atenuación
es el resultado.** Mide cuánto del efecto social pasa por la atención prenatal. Si no se atenuara,
significaría que la vía de las consultas no transmite nada del efecto educativo, lo que sería
un hallazgo en sí mismo.

**El error clásico, que este bloque existe para evitar.** Ajustar por mediadores y presentar
el coeficiente atenuado como «el efecto de la educación, ya controlado por todo». Eso
subestima el efecto total, invita a concluir que la educación importa menos de lo que importa,
y es exactamente el tipo de frase que un jurado subraya.

**La advertencia que acompaña a M4, y hay que declararla.** La descomposición en efecto total
y directo es válida bajo el supuesto de que **no hay confusores de la relación
mediador-desenlace**. Aquí ese supuesto es discutible: cualquier cosa que afecte a la vez al
número de controles y al peso al nacer —una complicación del embarazo, por ejemplo— lo
violaría. La descomposición se presenta como aproximada y se declara el supuesto.

In [ ]:
# ---------------------------------------------------------------------------
# M4: + mediador (consultas prenatales)
# ---------------------------------------------------------------------------

# NUMCONSUL viene con muchos valores distintos, incluidos algunos implausibles.
# Se agrupa en cuatro categorias con sentido clinico: la OMS recomienda al
# menos cuatro controles prenatales, y Colombia adopto ese umbral.
#
# POR QUE AGRUPAR Y NO USARLO COMO CONTINUO: el efecto no es lineal en el
# numero de consultas —la diferencia entre cero y cuatro no es la misma que
# entre diez y catorce— y tratarlo como continuo impondria esa linealidad.
consultas = pd.to_numeric(muestra["NUMCONSUL"], errors="coerce")
muestra["CONSULTAS4"] = pd.cut(
    consultas, bins=[-0.1, 0.5, 3.5, 6.5, 100],
    labels=["0 ninguna", "1 de 1 a 3", "2 de 4 a 6", "3 siete o mas"])

print("DISTRIBUCION DE LAS CONSULTAS PRENATALES\n")
dist = muestra["CONSULTAS4"].value_counts(dropna=False).sort_index()
for k, v in dist.items():
    print(f"  {str(k):<16} {v:>12,}  {v/len(muestra)*100:5.2f} %")

SOCIALES_MEDIADOR = SOCIALES_ETNIA + ["CONSULTAS4"]
ETIQUETAS["CONSULTAS4"] = {}

celdas_m4 = multinivel.agregar(muestra, SOCIALES_MEDIADOR, "MUNI_RES", "DPTO_RES")
X4, nombres_m4 = multinivel.matriz_diseno(celdas_m4, SOCIALES_MEDIADOR,
                                          ETIQUETAS, REFERENCIAS_FIJAS)

with cronometro("M4"):
    MODELOS["M4 +mediador"] = multinivel.ajustar(
        celdas_m4, X4, "MUNI_RES", "DPTO_RES", nombres_m4)

m4 = MODELOS["M4 +mediador"]
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

guardar_tabla(multinivel.tabla_efectos(m4), "multinivel_m4_mediador", decimales=4)
print("\nEFECTOS FIJOS\n")
print(multinivel.tabla_efectos(m4).to_string(index=False))

# --- La atenuacion: efecto total contra efecto directo ---------------------
#
# Se compara M4 contra M3 reajustado sobre la misma submuestra, por el mismo
# motivo que antes: CONSULTAS4 tiene nulos, de modo que M4 usa menos filas que
# M3 y la diferencia entre coeficientes mezclaria dos cosas.
sub4 = muestra.dropna(subset=["IDPERTET", "CONSULTAS4"] + AJUSTES)
celdas_m3b = multinivel.agregar(sub4, SOCIALES_ETNIA, "MUNI_RES", "DPTO_RES",
                                verbose=False)
X3b, nombres_m3b = multinivel.matriz_diseno(celdas_m3b, SOCIALES_ETNIA,
                                            ETIQUETAS, REFERENCIAS_FIJAS)
with cronometro("M3 sobre la submuestra de M4"):
    m3b = multinivel.ajustar(celdas_m3b, X3b, "MUNI_RES", "DPTO_RES",
                             nombres_m3b, verbose=False)
MODELOS["M3b (submuestra de M4)"] = m3b
pickle.dump(MODELOS, open(RUTA_MODELOS, "wb"))

comun = [n for n in m3b["nombres"] if n in m4["nombres"] and n != "intercepto"]
med = pd.DataFrame({
    "termino": comun,
    "rm_total": [np.exp(m3b["beta"][m3b["nombres"].index(n)]) for n in comun],
    "rm_directo": [np.exp(m4["beta"][m4["nombres"].index(n)]) for n in comun],
})
# Proporcion mediada, en escala logaritmica que es donde los efectos se suman.
# POR QUE EN LOGARITMOS: las razones de momios son multiplicativas; restar dos
# razones de momios no da la parte mediada. En escala log si.
b_tot = np.array([m3b["beta"][m3b["nombres"].index(n)] for n in comun])
b_dir = np.array([m4["beta"][m4["nombres"].index(n)] for n in comun])
with np.errstate(divide="ignore", invalid="ignore"):
    med["prop_mediada_pct"] = np.where(np.abs(b_tot) > 1e-6,
                                       (1 - b_dir / b_tot) * 100, np.nan).round(1)
med["atenua"] = np.abs(b_dir) < np.abs(b_tot)

guardar_tabla(med, "multinivel_mediacion", decimales=4)
print("\nEFECTO TOTAL FRENTE A EFECTO DIRECTO\n")
print(med.to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

educativos = med[med["termino"].str.startswith("EDUC4")]
print(f"\n  terminos educativos que se atenuan: "
      f"{int(educativos['atenua'].sum())} de {len(educativos)}")
print("  Si NO se atenuan, la via de las consultas prenatales no transmite el")
print("  efecto educativo, y eso es un hallazgo que hay que explicar, no un fallo.")

DISTRIBUCION DE LAS CONSULTAS PRENATALES

  0 ninguna             716,260   4.12 %
  1 de 1 a 3          2,043,321  11.76 %
  2 de 4 a 6          6,343,159  36.50 %
  3 siete o mas       7,199,893  41.43 %
  nan                 1,074,227   6.18 %
  filas originales       17,376,860
  descartadas             7,503,387  (43.18 %)
  celdas binomiales         921,670  (reduccion 11x)
  nacimientos por celda: mediana 1, maximo 78,289
[inicio] M4
  ajustando modelo de 3 niveles: 28 efectos fijos, 1123 MUNI_RES, 33 DPTO_RES
  convergio en 184 iteraciones, loglik = -567,641.4
[fin]    M4  ->  3435.3 min
tabla guardada: multinivel_m4_mediador.csv y multinivel_m4_mediador.tex

EFECTOS FIJOS

                                        termino  coeficiente  razon_momios
                                     intercepto      -1.6899        0.1845
  EDUC4_MADRE: Primaria vs Ninguno o preescolar      -0.0312        0.9693
EDUC4_MADRE: Secundaria vs Ninguno o preescolar      -0.0301        0.9703
  EDUC4_M

**Cómo se lee esta salida.**

**La columna `prop_mediada_pct` es el resultado del bloque.** Dice qué fracción del efecto
social pasa por la atención prenatal:

- **Positiva y entre el 5 % y el 40 %**: hay mediación apreciable. La atención prenatal es una
  de las vías por las que la posición social llega al peso al nacer, y por tanto una palanca
  de intervención.
- **Cercana a cero**: el efecto social opera por otras vías —nutrición, estrés, exposición
  laboral, ninguna de ellas registrada en las EEVV—, y mejorar el acceso a controles
  prenatales no cerraría la brecha.
- **Negativa**: el coeficiente creció al añadir el mediador. Indica que el supuesto de no
  confusión de la relación mediador-desenlace no se cumple, y la descomposición no es
  interpretable para ese término.

**Cómo citar los dos números sin confundirlos.** El efecto **total** es el que va a la frase
de política pública: *si todas las madres tuvieran educación secundaria o más, el momio de
bajo peso al nacer sería un X % menor*. El efecto **directo** es el que va a la discusión
sobre mecanismos: *de ese efecto, una fracción Y opera a través del acceso a atención
prenatal*.

Escribir la segunda cifra donde va la primera subestima el efecto de la educación, y es el
error que este bloque existe para evitar.

**El supuesto que hay que declarar, sin excepción.** La descomposición es válida si no hay
confusores de la relación entre consultas prenatales y bajo peso al nacer. Aquí ese supuesto
es discutible: una complicación del embarazo aumenta a la vez el número de controles y el
riesgo de bajo peso, y eso lo violaría —además de invertir el signo aparente de la mediación,
porque más controles aparecerían asociados a **más** riesgo. Si algún término educativo sale
con proporción mediada negativa, esa es la explicación más probable.

**Correspondencia con el manuscrito.** La tabla de mediación al Cuadro 5.7; la distinción
entre confusor y mediador, redactada en prosa, a la Sección 4.4.4 de la metodología; el
supuesto de no confusión, a las limitaciones.

## 6. Comparación de los modelos y diagnóstico

**Qué hacemos.** Reunimos las varianzas, el ICC, la MOR, el PCV y los criterios de
información de los cuatro modelos, y revisamos la convergencia y la distribución de los
efectos aleatorios.

**Qué responde el PCV.** Qué fracción de la variación territorial explica cada bloque de
variables:

$$\text{PCV}=\frac{\sigma^2_{\text{base}}-\sigma^2_{\text{modelo}}}{\sigma^2_{\text{base}}}$$

Un PCV del 40 % en el nivel municipal diría que el 40 % de las diferencias entre municipios
se debe a que sus poblaciones difieren en esas variables, y no a algo propio del municipio.

**Un PCV negativo no es un error.** Ocurre cuando las variables añadidas estaban distribuidas
de forma que enmascaraban parte de la variación territorial; al controlarlas, esa variación
se hace más visible. Hay que reportarlo como lo que es.

**Qué se revisa en el diagnóstico.** Que los cuatro modelos convergieran; que los efectos
aleatorios no tengan valores extremos que delaten municipios con denominadores diminutos; y
que su distribución sea aproximadamente simétrica, que es el supuesto del modelo.

In [ ]:
# ---------------------------------------------------------------------------
# Comparacion y diagnostico
# ---------------------------------------------------------------------------

PRINCIPALES = {k: v for k, v in MODELOS.items()
               if not k.startswith(("M1b", "M1c", "M2b", "M3b"))}
comparacion = multinivel.comparar(PRINCIPALES, "M0 vacio")
guardar_tabla(comparacion, "multinivel_comparacion", decimales=4)

print("COMPARACION DE LOS MODELOS\n")
print(comparacion.to_string(index=False, float_format=lambda x: f"{x:10.4f}"))

# --- Convergencia ----------------------------------------------------------
print("\n\nCONVERGENCIA\n")
for nombre, m in MODELOS.items():
    marca = "OK" if m["convergio"] else "NO CONVERGIO"
    print(f"  {nombre:<26} {marca:<14} {m['iteraciones']:>4} iteraciones  "
          f"loglik {m['loglik']:>14,.1f}")
    if not m["convergio"]:
        print(f"      {m['mensaje']}")

if not all(m["convergio"] for m in MODELOS.values()):
    print("\n" + "!"*66)
    print("ALGUN MODELO NO CONVERGIO. Sus coeficientes no son interpretables.")
    print("!"*66)

# --- Efectos aleatorios ----------------------------------------------------
v = m1["efectos_grupo"]
u = m1["efectos_supergrupo"]
print(f"\n\nEFECTOS ALEATORIOS DE M1\n")
print(f"  municipios    n={len(v):>5}  media {v.mean():+.4f}  sd {v.std(ddof=1):.4f}  "
      f"rango [{v.min():+.3f}, {v.max():+.3f}]")
print(f"  departamentos n={len(u):>5}  media {u.mean():+.4f}  sd {u.std(ddof=1):.4f}  "
      f"rango [{u.min():+.3f}, {u.max():+.3f}]")

# La media de los efectos debe rondar cero: es el supuesto del modelo, y una
# media lejos de cero indicaria que el intercepto no absorbio el nivel general.
if abs(v.mean()) > 0.05:
    print("  AVISO: la media de los efectos municipales se aleja de cero.")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

axes[0].hist(v, bins=50, color="tab:blue", alpha=0.8, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1.2)
axes[0].set_title(f"Efectos aleatorios municipales (M1)\nsd = {v.std(ddof=1):.3f}")
axes[0].set_xlabel("Efecto en escala logit")
axes[0].set_ylabel("Municipios")

# Diagrama cuantil-cuantil contra la normal: el supuesto del modelo es que los
# efectos son normales, y aqui se ve si lo son.
from scipy import stats
stats.probplot(v, dist="norm", plot=axes[1])
axes[1].set_title("Efectos municipales contra la normal")
axes[1].get_lines()[0].set_markersize(3)

# Varianza municipal a lo largo de los modelos: es la figura que resume la
# fase, porque muestra cuanta heterogeneidad territorial queda tras cada
# bloque de variables.
nombres_m = list(PRINCIPALES.keys())
var_m = [PRINCIPALES[n]["sigma_grupo"]**2 for n in nombres_m]
var_d = [PRINCIPALES[n]["sigma_supergrupo"]**2 for n in nombres_m]
x = np.arange(len(nombres_m))
axes[2].bar(x - 0.2, var_m, 0.4, label="Municipio", color="tab:blue")
axes[2].bar(x + 0.2, var_d, 0.4, label="Departamento", color="tab:orange")
for i, (a, b) in enumerate(zip(var_m, var_d)):
    axes[2].annotate(f"{a:.4f}", (i - 0.2, a), ha="center", va="bottom", fontsize=7.5)
    axes[2].annotate(f"{b:.4f}", (i + 0.2, b), ha="center", va="bottom", fontsize=7.5)
axes[2].set_xticks(x)
axes[2].set_xticklabels([n.split()[0] for n in nombres_m])
axes[2].set_ylabel("Varianza de los efectos aleatorios")
axes[2].set_title("Heterogeneidad territorial por modelo")
axes[2].legend()

fig.tight_layout()
guardar_figura(fig, "fig_multinivel_diagnostico")
plt.show()

print(f"\nmodelos guardados en {RUTA_MODELOS.name}")

**Cómo se lee esta salida.**

**La columna `convergio` se mira primero.** Un modelo que no converge no tiene coeficientes
interpretables, por razonables que parezcan.

**El ICC apenas se mueve entre modelos**, y es lo esperable: los determinantes sociales
explican variación **entre personas**, no entre territorios. Si el ICC cayera mucho al añadir
variables individuales, significaría que esas variables estaban capturando el efecto del
lugar, y habría que revisar la especificación.

**El PCV es la cifra que se cita.** Recordando que un valor negativo significa que la
variación territorial se hizo más visible al controlar la composición social, no que el
modelo falle.

**El diagrama cuantil-cuantil del panel central** comprueba el supuesto de normalidad de los
efectos aleatorios. Colas más pesadas que la normal son habituales y no invalidan el modelo;
lo que sí importa es detectar valores extremos aislados, que casi siempre corresponden a
municipios con muy pocos nacimientos. Con la mediana de 3.871 nacimientos por municipio, eso
no debería ocurrir aquí.

**El panel derecho es la figura que resume la fase**: cuánta heterogeneidad territorial queda
después de cada bloque de variables, en los dos niveles.

**Correspondencia con el manuscrito.** El cuadro comparativo es el Cuadro 5.4 completo; la
figura de diagnóstico al Apéndice; el panel derecho, al Capítulo 5.

## 7. Cierre

**Qué hacemos.** Listamos lo producido e imprimimos las versiones de las bibliotecas.

In [ ]:
# ---------------------------------------------------------------------------
# Cierre
# ---------------------------------------------------------------------------

from config import DIR_TABLAS, DIR_FIGURAS

print("RESUMEN DE LA FASE 4\n")
print(f"  modelos ajustados        {len(MODELOS)}")
print(f"  todos convergieron       {all(m['convergio'] for m in MODELOS.values())}")
print(f"  ICC municipal (M0)       {multinivel.icc(m0)['icc_grupo']*100:.2f} %")
print(f"  MOR municipal (M0)       {multinivel.mor(m0['sigma_grupo']):.3f}")
print(f"  nacimientos en M1        {m1['n_nacimientos']:,}")
print(f"  nacimientos en M3        {m3['n_nacimientos']:,}")
print(f"  nacimientos en M4        {m4['n_nacimientos']:,}")

print("\nTABLAS")
for nombre in ["multinivel_m1_social",
               "multinivel_m2_ajustado", "multinivel_m3_etnia", "multinivel_m4_mediador",
               "multinivel_cambio_por_etnia", "multinivel_efecto_del_ajuste",
               "multinivel_mediacion", "multinivel_comparacion"]:
    marca = "OK   " if (DIR_TABLAS / f"{nombre}.csv").exists() else "FALTA"
    print(f"  {marca}  {nombre}")

print(f"\nMODELOS  {RUTA_MODELOS.name}  "
      f"({RUTA_MODELOS.stat().st_size/1024:.0f} KB)")
for nombre in MODELOS:
    print(f"  {nombre}")

print("\n" + "=" * 60)
versiones()

## Qué sigue

**`05_descomposicion.ipynb`, Fase 5.** Descomposición de Wagstaff del índice de
concentración. Lee `intermedios/modelos_multinivel.pkl`, que este notebook acaba de dejar:
la descomposición necesita los coeficientes del modelo y las elasticidades de cada
determinante.

Quien reproduzca este análisis debe comprobar antes estos cinco puntos:

- [ ] Los cinco modelos convergieron
- [ ] El ICC municipal de M0 es **1,79 %** y la MOR municipal **1,184**
- [ ] Las razones de momios educativas de M1 son **menores que 1**, es decir el gradiente
      clásico, al contrario que en las Fases 1 y 3
- [ ] El PCV municipal de M1 es **negativo**, alrededor de −6 %
- [ ] En M2, edad materna en U, paridad por debajo de 1 y multiplicidad muy por encima
- [ ] Los coeficientes educativos **se atenúan** de M3 a M4

**El hallazgo que articula el Capítulo 5.** El gradiente educativo se invierte al controlar el
territorio: crudo apunta hacia las madres de mayor educación, ajustado apunta hacia las de
menor. La Fase 1 y la Fase 3 miden lo primero, esta fase lo segundo, y la diferencia entre
ambas **es** confusión territorial. Eso justifica el diseño multinivel de la tesis y hay que
presentarlo como el eje del capítulo, no como una nota técnica.

**Lo que queda declarado como limitación.** El eje étnico no es separable del territorial con
estos datos: con V de Cramér de 0,473 entre etnia y municipio, los interceptos aleatorios
absorben la mayor parte del efecto étnico antes de que el coeficiente fijo lo vea.

**Lo que falta y necesita una fuente externa.** El manuscrito contempla un modelo contextual
con el índice de pobreza multidimensional municipal. Esa variable no está en las EEVV y hay
que traerla del DANE, cruzada por código DIVIPOLA. Mientras no esté, el nivel municipal opera
como efecto aleatorio sin covariables contextuales, y eso se declara.